# 🛰️ SatQuery AI — Kaggle GPU Backend Server

Run this notebook on **Kaggle** with **GPU accelerator enabled** (Settings -> Accelerator -> GPU T4 x2 or P100) to host the real AI models for free!

---

### Step 1: Check GPU and Clone Codebase

In [ ]:
!nvidia-smi
!rm -rf novex
!git clone https://github.com/sanskar620/novex.git
%cd novex/backend

### Step 2: Install System and Python Dependencies

In [ ]:
# Install GDAL, PROJ, and cloudflared tunnel binary
!apt-get update -qq && apt-get install -y -qq gdal-bin libgdal-dev libgeos-dev libproj-dev > /dev/null
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Install backend dependencies with ML support (PyTorch + Transformers)
!pip install -q -e ".[dev]"
!pip install -q torch torchvision transformers accelerate peft

### Step 3: Configure Environment Variables (GPU Mode Enabled)

In [ ]:
import os

os.environ["DEMO_MODE"] = "false"
os.environ["ENVIRONMENT"] = "production"
os.environ["DEVICE"] = "cuda"
os.environ["AGENT_MODEL_NAME"] = "Qwen3-8B"
os.environ["CORS_ALLOWED_ORIGINS"] = "https://novex-teal.vercel.app,http://localhost:3000"

print("✅ Configuration set: DEMO_MODE=false, DEVICE=cuda (GPU Accelerated)")

### Step 4: Launch Public HTTPS Tunnel and Start Server
Run the cell below. It will output a public **`https://xxxx.trycloudflare.com`** URL. Copy that URL and paste it into Vercel!

In [ ]:
import subprocess
import threading
import time
import re

def monitor_tunnel():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*65)
            print("🚀 YOUR PUBLIC KAGGLE GPU BACKEND URL:")
            print(f"👉 {url}")
            print(f"👉 API URL for Vercel: {url}/api/v1")
            print("="*65 + "\n")

tunnel_thread = threading.Thread(target=monitor_tunnel, daemon=True)
tunnel_thread.start()
time.sleep(3)

# Start FastAPI Uvicorn Server
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000